# 💸 SpendSmart-ML — Final Empirical Research Orchestrator
## Google Colab GPU Edition

This notebook serves **only** as the execution orchestrator. All implementation logic exists in `src/` to ensure full reproducibility.

### Execution Modes:
- `smoke`: Rapid CPU-safe pipeline software verification. *Never used for paper claims.*
- `development`: GPU hyperparameter sweeps & ablations. Uses massive datasets.
- `final`: Immutable evaluation of the final chosen architecture. Triggering this checks `reports/final_model_manifest.json`.

In [ ]:
import os
import sys
import subprocess

REPO_URL = 'https://github.com/1Akash3/spendsmart-ml.git'
PROJECT_DIR = 'spendsmart-ml'

if REPO_URL and not os.path.isdir(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)

if not os.path.isdir(PROJECT_DIR):
    PROJECT_DIR = '.'

os.chdir(PROJECT_DIR)
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
print('✅ Working dir:', os.getcwd())

In [ ]:
!pip -q install -r requirements.txt
!pip -q install datasets pdfplumber huggingface_hub torch

In [ ]:
import pandas as pd
import json
import glob
from pathlib import Path
from IPython.display import display, Markdown

# Mode Selection
MODE = "smoke" # Options: smoke, development, final
print(f"🔥 CURRENT EXECUTION MODE: {MODE.upper()}")


## [Optional] Final Model Locking

If you are ready for the `final` execution, run the cell below to freeze your configuration. If you edit hyperparameters after this, the `final` test runner will crash!

In [ ]:
if MODE == "final":
    from src.locked_test_guard import LockedTestGuard
    # Freeze the architecture before the run
    LockedTestGuard.create_manifest({
        "architecture": "PATFormer",
        "sequence_length": 64,
        "embedding_dimension": 96,
        "layers": 3,
        "dropout": 0.15,
        "learning_rate": 0.001,
        "optimizer": "AdamW",
        "batch_size": 16,
        "preprocessing_version": "v1.0",
        "dataset_hash": "mocked_data_hash_for_now"
    })


## 1. Run Data Preprocessing & Splitting (Job 1)

In [ ]:
!python src/run_data_pipeline.py --mode {MODE}

## 2. Run Baselines & Categorization (Job 2 & 3)

In [ ]:
!python src/run_baselines.py --mode {MODE}

## 3. Run Neural Experiments (PATFormer & Ablations) (Job 4-9)

In [ ]:
!python src/run_neural_experiments.py --mode {MODE}

## 4. Render Final Tables from Persisted Artifacts

In [ ]:

reports_dir = Path(f"reports/results/{MODE}")
if not reports_dir.exists():
    print(f"No tables generated yet for mode: {MODE}!")
else:
    for table_file in sorted(reports_dir.glob("*.csv")):
        df = pd.read_csv(table_file)
        display(Markdown(f"### {table_file.stem} ({MODE.upper()})"))
        display(df)
